In [ ]:
#| hide
from kunda import *


# Kunda

Kunda selects Python interpreters, starts kernels, and keeps a bounded pool of them alive.

## Install

```sh
pip install kunda
```

## Select an interpreter

```python
from kunda import python_for, find_pythons, venv_env

python_for('~/code/myrepo/src', stop='~/code/myrepo')
find_pythons(roots=['~/code'], current=...)
venv_env(python)
```

`python_for` searches parent folders up to `stop`. It then uses the supplied default or a virtual environment under `roots`. `None` means the current interpreter.

`venv_env` removes frozen-host interpreter variables before starting the child.

## Run one kernel

```python
from kunda import Kernel

k = Kernel(cwd='~/code/myrepo', python=..., kernel='ipymini')
await k.start()
out = await k.execute('df.head()', on_output=print)
out.ok, out.text, out.execution_count
await k.complete('df.he', 5)
await k.restart()
await k.shutdown()
```

`ExecOutcome` contains execution errors and leaves the kernel available. `missing_kernel_module` reports a missing launcher after startup fails. Local kernels use `jupyter_client`; gateway kernels use the same interface over WebSocket.

## Keep several kernels alive

```python
from kunda import KernelPool, RuntimeBroker

pool = KernelPool(broker=RuntimeBroker(max_kernels=12), idle=30*60)
k = await pool.get('notebook-1', cwd=..., python=..., inspect=True)
pool.peek('notebook-1')
await pool.close('notebook-1')
await pool.close_all()
```

Each key has at most one kernel. The pool discards failed starts and keeps busy kernels. `idle=0` disables reaping.

`RuntimeBroker` enforces a process-wide limit. A host can also provide runners and known kernelspecs for other languages.

```python
KernelPool(runner_for=lambda lang: MyRustRunner if lang == 'rust' else None,
           known_kernels={'julia': 'julia-1.10'})
```

## Install kernel support

```python
from kunda import kernel_support, install_kernel_support, installable

kernel_support(python)
installable(python)
install_kernel_support(python)
```

Installation uses the host's installed package versions. It fails when either launcher still cannot import.

## Inspect live variables

With `inspect=True`, Kunda starts Dhrishti inside the kernel and reads its registry. A matching Python minor version uses the host package. Other versions use the project installation.